# Week 1 Project: Student Marks Dataset Analysis

**Student name:** ____________________  
**Registration number:** ____________________  
**Submission date:** ____________________

## Objective

Use Pandas and NumPy to clean the Student Marks dataset, calculate student averages, and display average, minimum, and maximum subject scores.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
DATA_PATH = Path('data/StudentsPerformance.csv')
RESULTS_PATH = Path('results')
RESULTS_PATH.mkdir(exist_ok=True)

## 1. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()

print('Dataset shape:', df.shape)
display(df.head())
display(df.dtypes.rename('data type').to_frame())
display(df.isna().sum().rename('missing values').to_frame())
print('Duplicate rows:', df.duplicated().sum())

## 2. Clean missing, duplicate, and invalid data

Categorical gaps are labeled `Unknown`. Invalid or missing subject scores are replaced with the median for that subject.

In [ ]:
required_columns = [
    'gender', 'race/ethnicity', 'parental level of education',
    'test preparation course', 'math score', 'reading score', 'writing score'
]
score_columns = ['math score', 'reading score', 'writing score']
category_columns = [
    'gender', 'race/ethnicity', 'parental level of education',
    'test preparation course'
]

missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f'Missing required columns: {missing_columns}')

original_rows = len(df)
duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

for column in category_columns:
    df[column] = df[column].astype('string').str.strip().replace('', pd.NA)

for column in score_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')
    df.loc[~df[column].between(0, 100), column] = np.nan

missing_before = df[required_columns].isna().sum()

for column in category_columns:
    df[column] = df[column].fillna('Unknown')

for column in score_columns:
    df[column] = df[column].fillna(df[column].median())

df['average score'] = np.round(
    np.mean(df[score_columns].to_numpy(dtype=float), axis=1), 2
)

print('Original rows:', original_rows)
print('Duplicate rows removed:', duplicates)
print('Rows after cleaning:', len(df))
display(missing_before.rename('values cleaned').to_frame())
display(df.head())

## 3. Calculate average, minimum, and maximum scores

In [ ]:
analyzed_columns = score_columns + ['average score']

score_summary = (
    df[analyzed_columns]
    .agg(['mean', 'min', 'max'])
    .transpose()
    .rename(columns={'mean': 'average', 'min': 'minimum', 'max': 'maximum'})
    .round(2)
)

display(score_summary)

## 4. Compare groups

In [ ]:
gender_summary = (
    df.groupby('gender')[analyzed_columns].mean().round(2)
)
preparation_summary = (
    df.groupby('test preparation course')[analyzed_columns].mean().round(2)
)
top_students = df.nlargest(5, 'average score')[
    ['gender', 'race/ethnicity', *score_columns, 'average score']
]

print('Average scores by gender')
display(gender_summary)
print('Average scores by test-preparation course')
display(preparation_summary)
print('Top five students')
display(top_students)

## 5. Save the cleaned data and result tables

In [ ]:
df.to_csv(RESULTS_PATH / 'cleaned_student_marks.csv', index=False)
score_summary.to_csv(RESULTS_PATH / 'score_summary.csv')
gender_summary.to_csv(RESULTS_PATH / 'gender_summary.csv')
preparation_summary.to_csv(RESULTS_PATH / 'preparation_summary.csv')
top_students.to_csv(RESULTS_PATH / 'top_students.csv', index=False)

print('All result files were saved in:', RESULTS_PATH.resolve())

## Conclusion

This project used Pandas to load, clean, group, and summarize student data. NumPy was used to calculate each student's average score. The final tables show the average, minimum, and maximum scores and allow performance comparisons by gender and test-preparation status.